# 06. Neural network: Multi-Layer Perceptron
**Run on:** Laptop B · **Approx. runtime:** ~6-8 min on a 4-core laptop


**Protocol (same for every model):**
- Each row is one *(GPS point, candidate arc)* pair. The label is 1 if the arc is the point's reference arc.
- **5-fold GroupKFold by trip** on the 78 usable training trips. Each fold fits on a 300k-row sample of points (KNN uses 100k) and is scored on all candidates of the held-out trips.
- The model is then **refit on the training split** and scored on the **10 student-test trips**.
- **Metrics:**
  - macro F1 / precision / recall: candidate level, threshold 0.5
  - log loss
  - **Top-1 accuracy**: the share of GPS points whose highest-scoring candidate is the reference arc
  - continuity
- The **HMM/Viterbi** decoder (parameters in `artifacts/hmm_params.json`) is then run on this model's probabilities to compare *classifier only* with *classifier + HMM*.


**Before running:** the dataset must be in `data/train/<id>/` and `data/student_test/<id>/`, and `pip install -r requirements.txt` done. Run all cells top to bottom (Kernel → Restart & Run All). Every number and plot below is computed when you run it.

In [ ]:
import os, sys, warnings
# run from notebooks/ or from the repo root
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
warnings.filterwarnings("ignore")
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160)
from pathsnap import nb
from pathsnap.models import make_model
from pathsnap.config import CFG
SEED = CFG["seed"]  # 42: every model and data split uses this seed
print("repo root:", ROOT)

In [ ]:
tr, tr_pts, te, te_pts = nb.load_data()

## MLP


**What it does:** a feed-forward network, 14 inputs → 64 ReLU → 32 ReLU → 1 sigmoid output, trained with Adam. Each layer learns non-linear combinations of the previous layer's outputs.
- **Why tried:** a different inductive bias from trees: smooth non-linear boundaries.
- **Limitations:** needs feature scaling, is sensitive to initialisation and learning rate, and is less interpretable.
- **Hyperparameters:** hidden layers (64, 32), alpha (L2) = 1e-4, Adam, early stopping on a 10% validation split, max 200 epochs.
- **Loss:** binary cross-entropy.

In [ ]:
make_mlp = lambda: make_model("mlp", SEED)
print(make_mlp())
cv_mlp = nb.run_cv(make_mlp, tr, tr_pts, max_rows=300000)
cv_mlp.round(4)

In [ ]:
model_mlp, train_mlp, test_mlp, p_mlp = nb.fit_and_evaluate(make_mlp, tr, tr_pts, te, te_pts, max_rows=300000)
print(f"student-test continuity (classifier only): {test_mlp['continuity']:.4f} | inference {test_mlp['infer_ms_per_point']:.4f} ms/point")
nb.summary_table("MLP", cv_mlp, train_mlp, test_mlp)

In [ ]:
imp_mlp = nb.plot_importance(model_mlp, te, "MLP")

In [ ]:
hmm_mlp = nb.hmm_compare(model_mlp, te, "MLP")
nb.plot_hmm(hmm_mlp, "MLP: classifier only vs classifier + HMM (student-test trips)")
nb.save_results("mlp", cv_mlp, train_mlp, test_mlp,
                {"hmm_mean": hmm_mlp.drop(columns=["trip", "model"]).mean().to_dict()})

### Training history (epoch-wise)
The MLP is the only model here that trains in epochs, so it is the only one with a genuine loss curve.

In [ ]:
mlp = model_mlp[-1]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(mlp.loss_curve_); ax[0].set(title="MLP training loss", xlabel="epoch", ylabel="log loss")
ax[1].plot(mlp.validation_scores_); ax[1].set(title="MLP validation accuracy (early-stopping split)", xlabel="epoch", ylabel="accuracy")
fig.tight_layout(); nb.show()
print(f"stopped after {mlp.n_iter_} epochs; best validation accuracy {max(mlp.validation_scores_):.4f}")

## Comparison of the models in this notebook

In [ ]:
nb.plot_cv_folds({"MLP": cv_mlp})
nb.plot_overfit({"MLP": (train_mlp, cv_mlp, test_mlp)})
nb.plot_confusion({"MLP": test_mlp["confusion"]})
nb.plot_calibration(te.label.to_numpy(), {"MLP": p_mlp})

In [ ]:
rows = []
for name, cv, t, h in [("MLP", cv_mlp, test_mlp, hmm_mlp)]:
    rows.append({"model": name, "CV macro F1": cv.macro_f1.mean(), "CV std": cv.macro_f1.std(),
                 "test Top-1": t["top1"], "test macro F1": t["macro_f1"], "test log loss": t["log_loss"],
                 "test continuity": t["continuity"], "Top-1 + HMM (trip mean)": h.hmm_top1.mean(),
                 "route F1 + HMM (trip mean)": h.hmm_route_f1.mean()})
pd.DataFrame(rows).set_index("model").round(4)